In [1]:
## The task is to analyse the 25 data files, summarise the data contained therein, and present 
## them in a report that will illustrate which genes and mutations the company should focus their efforts on.

## Each mutation experiment file will contain the following data in a tab-separated format:

## Gene name, Wild-type gene DNA sequence including promoter and coding region, Mutant DNA gene sequence 
## including promoter and coding region, Normalised cell viability – expressed as a percentage for the original 
## and mutant gene in 3 replicates, Normalised mRNA expression level measured by qRT-PCR for the original and mutant gene 
## in 3 replicates, Normalised protein expression level measured by a protein array for the original and mutant gene in 3 replicates.

## The analysis will complete the following tasks:

## Aggregate data in all files into a single data object in Python

## Using the data provided, identify the top 5 genes and mutations to prioritise validation experiments.

## Demonstrate that you have considered the type of mutation (substitution, insertion, deletion), 
## effect of the mutations on cell viability, promoter and coding sequences, 
## as well as how the protein and mRNA expression levels differ between wild-type and mutants.

## Plot the data to support your choice of the 5 genes. You can include up to 3 individual figures (a multi-panel graph counts as 1 figure), 
## one of which should match the example graph provided (note the exact data points and variables need not be the same).

In [35]:
import os
import numpy as np
import pandas as pd
import re
import glob

In [43]:
## This cell will import the folder of text files into python, read them, and enter them into a data frame

folder_path = "C:/Users/kathr/OneDrive - University of Aberdeen/BT5511 Report 2/Text Files"
## sets the folder path used for the code

if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")
## if the folder path does not exisat, inform user

txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

if not txt_files:
    raise ValueError(f"No .txt files found in: {folder_path}")
## confirms the presence of txt files in the folder

dataframe = []
for filename in txt_files:
    file_path = os.path.join(folder_path, filename)
    try:  
        df = pd.read_csv(file_path, sep="\t")
        if df.empty:
            print(f"Warning: {filename} is empty, skipping.")
            ## tells user if there are empty files
            continue
        dataframe.append(df)
    except Exception as e:
        print(f"Warning: could not read {filename} — {e}")
        ## skips txt files that cannot be read instead of crashing

combined_data = pd.concat(dataframe, ignore_index=True, sort=False)
combined_data

,Gene,WildType.Sequence,Mutant.Sequence,mRNA.Expression.WT.Rep1,mRNA.Expression.WT.Rep2,mRNA.Expression.WT.Rep3,mRNA.Expression.Mut.Rep1,mRNA.Expression.Mut.Rep2,mRNA.Expression.Mut.Rep3,Protein.Expression.WT.Rep1,...,Protein.Expression.WT.Rep3,Protein.Expression.Mut.Rep1,Protein.Expression.Mut.Rep2,Protein.Expression.Mut.Rep3,CellViability.WT.Rep1,CellViability.WT.Rep2,CellViability.WT.Rep3,CellViability.Mut.Rep1,CellViability.Mut.Rep2,CellViability.Mut.Rep3
0,Ah3,TGCCCGTCGTCAGAGGGTGGACGGTTACTATCAACGTCCGCTTCCA...,TGCCCGTCGTCAGAGGGTGGACGGTTACTATCAACGTCCGCTTCCA...,3.183135e+08,2.617708e+01,16.735333,3.183135e+08,2.776962e+01,17.761754,11685.000000,...,37123.000000,11693.000000,26726.000000,37133.000000,0.261116,0.726898,0.633374,0.338555,0.750318,0.710077
1,Alli8,TTTACCAGAATCTATAACAGTATATGGCAAAATTCTCCCGCGTACT...,TTTACCAGAATCTATAACAGTATATGGCAAAATTCTCCCGCGTACT...,6.389135e+01,5.165643e+00,13.914518,6.275914e+01,5.012645e+00,13.013207,14317.000000,...,42445.000000,14334.000000,4914.000000,42448.000000,0.062666,0.090101,0.256835,0.074965,0.163613,0.220555
2,Anap7c1,GGGCGCATAGTAGGCAAGACACCTATTGCTAAAAAACGTCCTGGAC...,GGGCGCATAGTAGGCAAGACACCTATTGCTAAAAAACGTCCTGGAC...,1.629126e+00,2.097780e+01,7.266207,-3.674294e+00,-1.094399e+02,-5.597223,25803.000000,...,4047.000000,8711.000000,21503.000000,0.000000,0.528095,0.765748,0.273578,0.000000,0.000000,0.000000
3,App1l8,ATCTTATCGTACACTATTAATAAACTAGCGCTGATTCGAGTCGCTC...,ATCTTATCGTACACTATTAATAAACTAGCGCTGATTCGAGTCGCTC...,3.619328e+00,7.207154e+00,1.347365,3.898224e+00,6.585581e+00,0.413255,44784.000000,...,3517.000000,44785.000000,32607.000000,3530.000000,0.412541,0.059328,0.476712,0.184925,0.017589,0.460078
4,Avon4,TGAATCTAAGATTAAGTGGATATCGCCTTGACTTCTTTATCCATCC...,TGAATCTAAGATTAAGTGGATATCGCCTTGACTTCTTTATCCATCC...,1.398400e+00,1.745494e+00,4.394474,2.973012e+00,2.412630e+00,6.489425,29811.000000,...,45335.000000,54659.000000,55693.000000,55698.000000,0.829231,0.301918,0.795280,5.874721,6.243796,5.455045
5,Cairn3a2,GCACTATGGAACAAATCTCCGTAGGATAGCCAGAGTAAATCGGCCT...,GCACTATGGAACAAATCTCCGTAGGATAGCCAGAGTAAATCGGCCT...,2.529018e+01,2.711265e+00,48.099042,2.539605e+01,2.666513e+00,49.186680,25992.000000,...,31383.000000,26001.000000,27949.000000,31399.000000,0.633514,0.956889,0.518404,0.664322,0.970643,0.692276
6,Cairn6,GATTGCTAGTGGTAGTGGTGGCCCGGGTCGCCGTCCGCCACTCTTT...,GATTGCTAGTGGTAGTGGTGGCCCGGGTCGCCGTCCGCCACTCTTT...,2.084163e+01,3.676616e+01,5213.843993,2.274574e+01,3.577278e+01,5213.722502,48979.000000,...,27149.000000,48992.000000,33458.000000,27163.000000,0.925855,0.799526,0.801135,0.946808,0.806410,0.835041
7,Clach6,AAATGACATCCTCCGGGGAACGATGCAGGCCCAGTAAGCCAATACT...,AAATGACATCCTCCGGGGAACGATGCAGGCCCAGTAAGCCAATACT...,1.985791e+01,3.451715e+00,2.631587,1.783861e+01,3.045175e+00,3.017050,48988.000000,...,7210.000000,48994.000000,4939.000000,7221.000000,0.902357,0.554833,0.911485,0.884963,0.605348,1.001523
8,Dsgt9a1,ATAATCGCTGGTGTACTTCAAGTTACGACTTGAGTACGCATATCCA...,ATAATCGCTGGTGTACTTCAAGTTACGACTTGAGTACGCATATCCA...,9.898111e+00,2.371426e+00,101.458408,2.041802e+01,3.124716e+00,101.491889,25570.000000,...,10606.000000,41200.000000,42018.000000,25348.000000,0.637916,0.149896,0.124796,0.000000,0.000000,0.000000
9,Ever6b6,AAGTTATTCTACGGGCCCTCTATTGAATAGATAACCCAGGGGCGTG...,AAGTTATTCTACGGGCCCTCTATTGAATAGATAACCCAGGGGCGTG...,3.167661e+00,3.178356e+06,3.828825,2.268305e+00,3.178356e+06,2.285579,10686.000000,...,48037.000000,10695.000000,639.000000,48051.000000,0.108788,0.568027,0.022040,0.286469,0.673888,0.136360


In [47]:
## This cell will identify type of mutation present in each mutant sequence when compared to refernce sequence

VALID_BASES = set("ATCG")
## sets the valid DNA bases and A, C, T, and G

def validate_inputs(df, col2, col3):
    for col in [col2, col3]:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found. Available columns: {df.columns.tolist()}")
    ## this validates that the columns set by the code exist in the data frame
    
    if df.empty:
        raise ValueError("DataFrame is empty.")
    ## this confirms that the data frame is not empty

def validate_sequence(seq, row_index, col_name):
    ## validates that the sequences only contain valid DNA bases
    
    invalid = set(seq) - VALID_BASES
    if invalid:
        raise ValueError(f"Row {row_index}, column '{col_name}' contains invalid bases: {invalid}")

def compare_dna_sequences(df, col2, col3):

    df.columns = df.columns.str.strip()
    ## get rid of whitespace in the columns

    results = []
    skipped_rows = []

    for index, row in df.iterrows():

        if pd.isna(row[col2]) or pd.isna(row[col3]):
            print(f"Warning: Row {index} contains missing sequence(s), skipping.")
            skipped_rows.append(index)
            continue
            ## tells the user if there are missing sequences in any of the columns

        seq1 = str(row[col2]).strip().upper()
        seq2 = str(row[col3]).strip().upper()
        ## strips any white space around the sequence and confirms every character is uppercase

        if len(seq1) == 0 or len(seq2) == 0:
            print(f"Warning: Row {index} contains empty sequence(s), skipping.")
            skipped_rows.append(index)
            continue
            ## another checkpoint to tell the user if there is a missing sequence

        try:
            validate_sequence(seq1, index, col2)
            validate_sequence(seq2, index, col3)
        except ValueError as e:
            print(f"Warning: {e}, skipping.")
            skipped_rows.append(index)
            continue
        ## validate the sequences again

        len1 = len(seq1)
        len2 = len(seq2)
        len_diff = len1 - len2
        ## calculate the difference in length between wildtype and mutation

        ## determine deletion or insertion
        if len_diff > 0:
            mutation_type = "Deletion"
            mutation_size = len_diff
            ## wildtype is longer so there is a deletion in mutated seq

            mutation_start = None
            for i, (base1, base2) in enumerate(zip(seq1, seq2)):
                if base1 != base2:
                    mutation_start = i + 1
                    break

            if mutation_start is None:
                mutation_start = len2 + 1

            affected_bases = seq1[mutation_start - 1: mutation_start - 1 + mutation_size]
            mutation_label = "Deleted bases"

            if mutation_size % 3 == 0:
                frameshift = "No — in-frame deletion"
                codon_effect = f"Deletion of {mutation_size} base(s) removes {mutation_size // 3} complete codon(s)"
                ## deletion removes at least one entire codon
            else:
                frameshift = "Yes — frameshift mutation"
                codon_effect = f"Deletion of {mutation_size} base(s) shifts reading frame by {mutation_size % 3} position(s)"
                ## deletion causes a frameshift

        elif len_diff < 0:
            mutation_type = "Insertion"
            mutation_size = abs(len_diff)
            ## mutated sequence is longer than wt

            mutation_start = None
            for i, (base1, base2) in enumerate(zip(seq1, seq2)):
                if base1 != base2:
                    mutation_start = i + 1
                    break

            if mutation_start is None:
                mutation_start = len1 + 1

            affected_bases = seq2[mutation_start - 1: mutation_start - 1 + mutation_size]
            mutation_label = "Inserted bases"

            if mutation_size % 3 == 0:
                frameshift = "No — in-frame insertion"
                codon_effect = f"Insertion of {mutation_size} base(s) adds {mutation_size // 3} complete codon(s)"
                ## insertion removes at least one entire codon
            else:
                frameshift = "Yes — frameshift mutation"
                codon_effect = f"Insertion of {mutation_size} base(s) shifts reading frame by {mutation_size % 3} position(s)"
                ## insertion causes frameshift 

        else:
            mutation_type = "Substitution"
            mutation_size = 0
            mutation_start = None
            affected_bases = None
            mutation_label = "Affected bases"
            frameshift = "No — same length, substitutions only"
            codon_effect = "No change in reading frame"

        ## position by position comparison
        max_len = max(len1, len2)
        seq1_padded = seq1.ljust(max_len, "-")
        seq2_padded = seq2.ljust(max_len, "-")

        mismatches = []
        diff_marker = []
        for i, (base1, base2) in enumerate(zip(seq1_padded, seq2_padded)):
            if base1 != base2:
                mismatches.append(f"pos {i+1}: {base1}->{base2}")
                diff_marker.append("^")
            else:
                diff_marker.append(".")

        alignment = "".join(diff_marker)
        mismatch_count = len(mismatches)
        percent_identity = round(((max_len - mismatch_count) / max_len) * 100, 2)

        results.append({
            "row_index":        index,
            col2:               seq1,
            col3:               seq2,
            "alignment":        alignment,
            "mismatches":       ", ".join(mismatches) if mismatches else "None",
            "mismatch_count":   mismatch_count,
            "percent_identity": f"{percent_identity}%",
            "mutation_type":    mutation_type,
            "mutation_size":    mutation_size,
            "mutation_start":   mutation_start,
            mutation_label:     affected_bases,
            "frameshift":       frameshift,
            "codon_effect":     codon_effect
        })

        print(f"\n--- Row {index} ---")
        print(f"  Wildtype Sequence: {seq1}")
        print(f"  Mutated Sequence:     {seq2}")
        print(f"  Alignment: {alignment}")
        print(f"  Mismatches: {', '.join(mismatches) if mismatches else 'None'}")
        print(f"  Percent Identity: {percent_identity}%")
        print(f"  Mutation Type: {mutation_type}")
        if mutation_size > 0:
            print(f"  Mutation Size: {mutation_size} base(s)")
            print(f"  Mutation Start Position: {mutation_start}")
            print(f"  {mutation_label}: {affected_bases}")
        print(f"  Frameshift: {frameshift}")
        print(f"  Codon Effect: {codon_effect}")

    ## final summary
    if not results:
        raise ValueError("No valid rows were processed. Check your data for missing or invalid sequences.")
    
    print(f"\n=== Summary ===")
    print(f"  Rows processed:  {len(results)}")
    print(f"  Rows skipped:    {len(skipped_rows)}")
    if skipped_rows:
        print(f"  Skipped row indices: {skipped_rows}")

    return pd.DataFrame(results)

diff_df = compare_dna_sequences(combined_data, "WildType.Sequence", "Mutant.Sequence")
print(diff_df)


--- Row 0 ---
  Wildtype Sequence: TGCCCGTCGTCAGAGGGTGGACGGTTACTATCAACGTCCGCTTCCAAGCAGTCGGGCTTCAGGAACACCGTAGGCTTCTTTTGCCGCAGTTGGGCTCTACAACGGCGCAGAAAATGTACGGTCATAAACGCTGCACCTATAACCAACATCGTCACTAACCCTTTTGCGGGTCGTAGATTAAAAACGCTGACGGCAGTACGGCGTCTCTAGACGCCCGGGGGCAAATAAACATGGTCGATCAATGACGCCGTCAATCAACGGCAATAAAGTAGTTGGCCAGGTGCGAGTCACATCGGTGTCGAGTGTTCTTCCTCTCGCGTCCCGATGCGCAGCACGACTTCAAAGGTATTGGTTCCGTCATGCAACGTCGGATATCGAGTCACACTGTATTACCAAATTCTACTTTACCAGAGATATATAAGATGGAATACGCTATTTGTCACATAGTGTCCGATAACGTTGAAGTTAGATACATGCCGTACTGTCCCGTTTTTAAAGCCTGTCTTTGCGAGTGATCCAGTTGGTGCCGATAGTCATATTTCGCCCACAGAATCCCTTTACTGCACCGTCAGACAGTGAAATAGCAGGCGAAGCTAGGGTCCATTTTTTTTCGTATGCGTCGGGTACGCCGGAGGACGCGCAAGCTGCTTTTAGTCGAGGTCAACTTATGTCACGCGTGCTTCGTCTGCATTCCAGGATTAACTTCTGCACATACACAGAAAGCAAGAATGCCAAACGCATCCTTCCTTACCGGCTAGTGAGCCTACGCTTTCGCTGCTCTGACCTACTTCCTGGCGTTATGAGGGCTCCTGGCTCCTGCGACGGCGATGCCGTGTGGTACCAATCAAACTGCGGGTAGGTTCGAGGTTCGCATCAGACTAACCTCCGTTCCCACACGGGTTATTACACGAATAACGGCGACTCCATGGTCGCTCAACCCCCTTACAAGATTTAAATAAATGAA

In [ ]:
## print(combined_data['Gene'].loc[combined_data.index[0]])
## this will give you information in a specific cell

## Column Names in combined_data

## Gene

## WildType.Sequence

## Mutant.Sequence

## mRNA.Expression.WT.Rep1
## mRNA.Expression.WT.Rep2
## mRNA.Expression.WT.Rep3

## mRNA.Expression.Mut.Rep1
## mRNA.Expression.Mut.Rep2
## mRNA.Expression.Mut.Rep3

## Protein.Expression.WT.Rep1
## Protein.Expression.WT.Rep2
## Protein.Expression.WT.Rep3

## Protein.Expression.Mut.Rep1
## Protein.Expression.Mut.Rep2
## Protein.Expression.Mut.Rep3

## CellViability.WT.Rep1
## CellViability.WT.Rep2
## CellViability.WT.Rep3

## CellViability.Mut.Rep1
## CellViability.Mut.Rep2
## CellViability.Mut.Rep3


## for item in combined_data:
   ## larger_string = item[whildtype.sequence]
   ## short_string = other one
   ## if len(larger_string) > len(item[mutate.sequence])
    ##    largre_string = item[mutate.sequence]
   ##     shot_string = other one


    ##    char_long = larger_string[0]
    ##    char_short = short_string[0]

 ##   for i in range(len(long_string)):
     ##   if char_long == char_short:
      ##      char_long = larger_string[i]
       ##     char_short = short_string[i]
       ##     continue
    ##    else:

In [ ]:
print(combined_data['WildType.Sequence'].loc[combined_data.index[0]])

In [ ]:
for index, row in combined_data.iterrows():
    
    for nt in range(len(combined_data['WildType.Sequence'].loc[combined_data.index[0]])):
    
        seq1 = []
        seq1.append()
        
    for idx2 in range(len(combined_data['Mutant.Sequence'].loc[combined_data.index[0]])):
            seq2 = (combined_data['Mutant.Sequence'].loc[combined_data.index[0]])[idx2]

            for nt in range(len(seq1)):
                if seq1[nt] == seq2[nt]:
                    print(nt)
                else:
                    print("no")
        

In [ ]:
for 




for idx1 in range(len(combined_data['WildType.Sequence'].loc[combined_data.index[0]])):
    # loop 1 gets each seq
    # loop 2 gets the other seq to compare to.
    seq1 = (combined_data['WildType.Sequence'].loc[combined_data.index[0]])[idx1]
    for idx2 in range(len(combined_data['Mutant.Sequence'].loc[combined_data.index[0]])):
        seq2 = (combined_data['Mutant.Sequence'].loc[combined_data.index[0]])[idx2]
        # assume the sequences are the same length
        for nt in range(len(seq1)):
            if seq1[nt] == seq2[nt]:
                dna_array[idx1, idx2, nt] = 1
            else:
                dna_array[idx1, idx2, nt] = 0 # we don't actually need this because the default value is already 0

In [ ]:
## iterate over each wildtype and compare to mutant

for seq in range(len(dna_seqs)):
        # will give numbers
        # iterate over the five DNA sequences

        dna_seq_str = dna_seqs[seq]
        # indexes specific sequence 
        
        for nt in range(len(dna_seqs[0])):
        # makes the string ACGT an integer
    
            nt_str = (dna_seqs[0])[nt]
            # inside round brackets to find first element of list DNA sequences then find specific nucleotide at position nt
            # want first operation before the indexing operation
            # looping over nucleotides
    
            othernt_str = dna_seq_str[nt]
    
            if nt_str == othernt_str:
                # want to know if nucleotide at that position is the same or not
                # compare the strings
                # actual nucleotides at those positions have been extracted
                
                dna_array[seq, nt] = 1
                # looking at what sequence you're in
                # need to define all the dimensions you have

            else:
                dna_array[seq, nt] = 0

In [ ]:
## This cell will identify type of mutation

In [ ]:
## This cell will identify effect on cell viability, promoter, and coding sequences

In [ ]:
## This cell will identify protein and mRNA expression levels between wild-type and mutant

In [ ]:
## This cell will identify top 5 genes and mutations

In [ ]:
## This cell will plot data to support choice of 5 genes and mutations to prioritise